# 第 9 章 財政政策と金融政策の効果: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/09-fiscal-and-monetary-policy.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
mpc = 0.6                # 限界消費性向 c
b = 20                   # 投資の利子感応度 b
intercept = 450          # IS 曲線の A
k = 0.5                  # 貨幣需要の所得感応度 k
h = 100                  # 貨幣需要の利子感応度 h
real_money = 250         # 実質貨幣供給 M/P
delta_government = 10    # 政府支出の増加 ΔG
delta_real_money = 10    # 実質貨幣供給の増加 Δ(M/P)
trap_interest = 2.5      # 流動性のわなの、利子率の下限

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 政策の効果

In [ ]:
from lib.chapter09.policy import (
    crowding_out,
    fiscal_effect,
    fiscal_effect_solved,
    monetary_effect,
    monetary_effect_solved,
)

fiscal = fiscal_effect(mpc, b, k, h, delta_government)
fiscal_solved = fiscal_effect_solved(
    mpc, b, intercept, k, h, real_money, delta_government
)
monetary = monetary_effect(mpc, b, k, h, delta_real_money)
monetary_solved = monetary_effect_solved(
    mpc, b, intercept, k, h, real_money, delta_real_money
)
print(f"財政政策の効果 ΔY: {fiscal:.2f}（連立方程式を解くと {fiscal_solved:.2f}）")
print(f"金融政策の効果 ΔY: {monetary:.2f}（連立方程式を解くと {monetary_solved:.2f}）")
print(f"クラウディングアウト: {crowding_out(mpc, b, k, h, delta_government):.2f}")

## 条件ごとの比較（h が無限大 = 流動性のわな）

In [ ]:
import math

from lib.chapter09.policy import policy_effectiveness

conditions = [(h, b), (10, b), (1000, b), (math.inf, b)]
rows = policy_effectiveness(mpc, k, conditions, delta_government, delta_real_money)
for row in rows:
    print(f"h={row.h}, b={row.b}: 財政 {row.fiscal:.2f}、金融 {row.monetary:.2f}、"
          f"クラウディングアウト {row.crowding_out:.2f}")

## 図: 政策による曲線のシフト

In [ ]:
from lib.chapter09.plot import plot_policy_shifts

plot_policy_shifts(
    mpc, b, intercept, k, h, real_money, delta_government, delta_real_money,
    income_min=600, income_max=1200,
)

## 図: 流動性のわな

In [ ]:
from lib.chapter09.plot import plot_liquidity_trap

plot_liquidity_trap(
    mpc, b, intercept, trap_interest, delta_government, income_min=600, income_max=1200
)

## 試してみよう

- 条件の表の `h = 10`（貨幣需要が利子に鈍い）では、財政 7.14、金融 14.29、
  クラウディングアウト 17.86 です（`h = 100` では、20.00、4.00、5.00）。
  LM 曲線が急だと、財政政策は効きにくく、金融政策が効きます
- `h = 1000` では、財政 24.39、金融 0.49 です。LM 曲線が水平に近いと、
  財政政策がほぼ単純な乗数どおりに効き、金融政策は効きません。
  `h = math.inf`（流動性のわな）では、財政 25.00（= 1 ÷ (1 − c) × 10）、金融 0.00 です
- `b = 40` にすると、財政政策の効果は 16.67 に小さくなります（`b = 20` では 20.00）。
  投資が利子に敏感だと、クラウディングアウトが大きくなります
- 「連立方程式を解くと」の値が、閉じた式の値と一致します（財政 20.00、金融 4.00）